# Kaggriculture | Adaptive Farm Intelligence — Fieldbook Route Refresh

A compact Kaggriculture submission package.


## Method

The proven Fieldbook opening and all six recorded action tapes are unchanged. At the day-9 terminal branch only, cash, land and wheat price are read from the current observation rather than the stale day-6 snapshot. The notebook writes the required `submission.tar.gz`.


In [ ]:
# Self-contained competition package with a narrow day-9 route refresh.
import base64
import hashlib
import io
import tarfile
from pathlib import Path

ARCHIVE_B64 = (
    '...'
    '/Xz6+fTz6efTz6efTz//tX7+Lx9gqtQAKAUA'
)
EXPECTED_ARCHIVE_SHA256 = 'dfa6e71251493e47380b5acbcbabe70d48748c382f1b586926d6ea56af7deb12'
EXPECTED_AGENT_SHA256 = '71bb7c8fcd2470dbbbf956df18061e5f8024f2924c96f11060e72c968ab638f1'

payload = base64.b64decode(ARCHIVE_B64)
assert hashlib.sha256(payload).hexdigest() == EXPECTED_ARCHIVE_SHA256
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    source = archive.extractfile("main.py").read()
    assert hashlib.sha256(source).hexdigest() == EXPECTED_AGENT_SHA256
    compile(source, "main.py", "exec")
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("archive sha256:", EXPECTED_ARCHIVE_SHA256)
print("agent sha256:", EXPECTED_AGENT_SHA256)
